
# Assignment 4 — Training a DCGAN on the Fashion-MNIST Dataset

**Objective:** Understand how a Deep Convolutional Generative Adversarial Network (DCGAN) is trained and how the generator and discriminator improve over time.

This notebook implements a DCGAN in **PyTorch** on the Fashion-MNIST dataset. It:
- Uses transposed convolution layers in the Generator.
- Uses convolution layers in the Discriminator.
- Trains for **30 epochs**.
- Saves a **4 × 4 grid** of generated images every 5 epochs.
- Records Generator and Discriminator losses.
- Plots both losses after training.
- Provides observations about image quality, recognisable clothing shapes, saturation, and possible instability/mode collapse.


In [ ]:

# 1. Import libraries and set configuration

import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
BATCH_SIZE = 128
LATENT_DIM = 100
IMAGE_SIZE = 28
CHANNELS = 1
NUM_EPOCHS = 30
LR = 0.0002
BETA1 = 0.5
BETA2 = 0.999

OUTPUT_DIR = "dcgan_fashion_mnist_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Output directory:", os.path.abspath(OUTPUT_DIR))



## 2. Load and preprocess Fashion-MNIST

The images are grayscale clothing images of size 28 × 28. Pixel values are normalized to approximately [-1, 1], which matches the `Tanh` activation used at the Generator output.


In [ ]:

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Number of training images:", len(train_dataset))
print("Number of batches per epoch:", len(train_loader))


In [ ]:

# Display a few real Fashion-MNIST images

images, labels = next(iter(train_loader))
images_display = (images[:16] + 1) / 2

grid = make_grid(images_display, nrow=4, padding=2)
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("Sample Fashion-MNIST Images")
plt.axis("off")
plt.show()



## 3. Define the Generator

The Generator takes a random latent vector of length 100 and transforms it into a 28 × 28 grayscale image. Transposed convolution layers are used to upsample the representation.


In [ ]:

class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()

        self.model = nn.Sequential(
            # Input: latent_dim x 1 x 1
            nn.ConvTranspose2d(latent_dim, 128, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # 7 x 7 -> 14 x 14
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            # 14 x 14 -> 28 x 28
            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z)

G = Generator(LATENT_DIM).to(DEVICE)
print(G)



## 4. Define the Discriminator

The Discriminator receives a 28 × 28 grayscale image and uses convolutional layers to determine whether the image is real or generated.


In [ ]:

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            # 28 x 28 -> 14 x 14
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 14 x 14 -> 7 x 7
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # 7 x 7 -> 1 x 1
            nn.Conv2d(128, 1, kernel_size=7, stride=1, padding=0, bias=False)
        )

    def forward(self, x):
        return self.model(x).view(-1)

D = Discriminator().to(DEVICE)
print(D)


In [ ]:

# Weight initialization recommended for DCGANs

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

G.apply(weights_init)
D.apply(weights_init)

criterion = nn.BCEWithLogitsLoss()

optimizer_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2))
optimizer_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, BETA2))

print("Models initialized and optimizers created.")



## 5. Training setup

A fixed latent-noise batch is used for the saved image grids. This makes it easier to compare the Generator's progress from epoch to epoch.

The labels are:
- **Real = 1**
- **Fake = 0**


In [ ]:

fixed_noise = torch.randn(16, LATENT_DIM, 1, 1, device=DEVICE)

G_losses = []
D_losses = []
epoch_G_losses = []
epoch_D_losses = []
saved_epochs = []

def save_generated_grid(epoch):
    G.eval()
    with torch.no_grad():
        fake = G(fixed_noise).detach().cpu()
    fake = (fake + 1) / 2
    path = os.path.join(OUTPUT_DIR, f"generated_epoch_{epoch:02d}.png")
    save_image(fake, path, nrow=4, padding=2)
    G.train()
    return path

print("Training will run for", NUM_EPOCHS, "epochs.")



## 6. Train the DCGAN

For every batch:
1. The Discriminator learns to distinguish real images from fake images.
2. The Generator learns to produce images that fool the Discriminator.
3. Both losses are recorded.

A generated-image grid is saved after epochs **5, 10, 15, 20, 25, and 30**.


In [ ]:

for epoch in range(1, NUM_EPOCHS + 1):
    running_G = 0.0
    running_D = 0.0
    num_batches = 0

    for real_images, _ in train_loader:
        real_images = real_images.to(DEVICE)
        batch_size = real_images.size(0)

        real_targets = torch.ones(batch_size, device=DEVICE)
        fake_targets = torch.zeros(batch_size, device=DEVICE)

        # -------------------------
        # Train Discriminator
        # -------------------------
        optimizer_D.zero_grad()

        real_logits = D(real_images)
        d_real_loss = criterion(real_logits, real_targets)

        z = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
        fake_images = G(z)
        fake_logits = D(fake_images.detach())
        d_fake_loss = criterion(fake_logits, fake_targets)

        d_loss = d_real_loss + d_fake_loss
        d_loss.backward()
        optimizer_D.step()

        # -------------------------
        # Train Generator
        # -------------------------
        optimizer_G.zero_grad()

        fake_logits_for_G = D(fake_images)
        g_loss = criterion(fake_logits_for_G, real_targets)

        g_loss.backward()
        optimizer_G.step()

        G_losses.append(g_loss.item())
        D_losses.append(d_loss.item())

        running_G += g_loss.item()
        running_D += d_loss.item()
        num_batches += 1

    avg_G = running_G / num_batches
    avg_D = running_D / num_batches

    epoch_G_losses.append(avg_G)
    epoch_D_losses.append(avg_D)

    if epoch % 5 == 0:
        path = save_generated_grid(epoch)
        saved_epochs.append(epoch)
        print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] | G Loss: {avg_G:.4f} | D Loss: {avg_D:.4f}")
        print("Saved:", path)
    else:
        print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] | G Loss: {avg_G:.4f} | D Loss: {avg_D:.4f}")

print("\nTraining complete.")



## 7. Plot Generator and Discriminator losses

GAN losses do not behave like ordinary supervised-learning losses. They can oscillate because the Generator and Discriminator are continuously adapting to one another. Therefore, the overall trend and generated images should be considered together.


In [ ]:

plt.figure(figsize=(10, 5))
plt.plot(range(1, NUM_EPOCHS + 1), epoch_G_losses, label="Generator Loss")
plt.plot(range(1, NUM_EPOCHS + 1), epoch_D_losses, label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DCGAN Training Losses on Fashion-MNIST")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

loss_path = os.path.join(OUTPUT_DIR, "loss_curves.png")
plt.savefig(loss_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved loss graph to:", loss_path)



## 8. Display generated image grids from different epochs


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(10, 7))

for ax, epoch in zip(axes.flat, saved_epochs):
    path = os.path.join(OUTPUT_DIR, f"generated_epoch_{epoch:02d}.png")
    img = plt.imread(path)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"Epoch {epoch}")
    ax.axis("off")

plt.suptitle("Generated Fashion-MNIST Images Over Training", fontsize=14)
plt.tight_layout()

comparison_path = os.path.join(OUTPUT_DIR, "generated_epochs_comparison.png")
plt.savefig(comparison_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved comparison grid to:", comparison_path)



## 9. Training observations

### Early epochs
During the early epochs, the Generator normally produces noisy, blurry, or poorly structured images. The outputs may contain vague edges or silhouettes, but they are usually not recognisable as specific clothing categories.

### Development of clothing shapes
As training continues, the Generator learns common visual patterns from Fashion-MNIST, such as horizontal/vertical edges, sleeves, shoe-like shapes, shirts, bags, and other clothing silhouettes. Recognisable shapes should begin to appear around the middle portion of training, although the exact epoch can vary between runs and hardware.

### Later epochs
By the later epochs, the generated samples should generally have clearer outlines and more coherent clothing-like structures. They will still not be as sharp or detailed as real Fashion-MNIST images because the dataset is small and the model is deliberately compact.

### Does image quality stop improving?
GAN training often reaches a point where improvements become smaller. The Generator and Discriminator can enter a dynamic equilibrium in which the images fluctuate in quality instead of improving smoothly at every epoch. Therefore, a later epoch is not guaranteed to look better than every earlier epoch.

### Mode collapse / instability
Mode collapse occurs when the Generator produces very similar samples and fails to represent the diversity of the training data. Inspect the 4 × 4 grids for repeated or nearly identical clothing shapes. Some variation in quality or loss values is normal in GAN training and does not necessarily indicate failure.

### Overall conclusion
The experiment demonstrates the competitive nature of GAN training. The Discriminator learns to distinguish real Fashion-MNIST images from generated images, while the Generator simultaneously learns to create increasingly realistic clothing-like images. The loss curves may oscillate rather than decrease monotonically, so visual inspection of the saved image grids is an important part of evaluating training progress.


In [ ]:

# 10. Print a compact summary for the assignment report

print("DCGAN Training Summary")
print("-" * 40)
print(f"Dataset: Fashion-MNIST")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Latent dimension: {LATENT_DIM}")
print(f"Learning rate: {LR}")
print(f"Saved image epochs: {saved_epochs}")
print(f"Final Generator loss: {epoch_G_losses[-1]:.4f}")
print(f"Final Discriminator loss: {epoch_D_losses[-1]:.4f}")
print(f"Outputs saved in: {os.path.abspath(OUTPUT_DIR)}")
